
# 🛡️ Drift Detective — AURORA-QR (2025) — No-Code Lab

**Scenario:** Attackers use AI to write polished emails, hide links in **QR codes**, and use **voice clones** to call the help desk.  
Today looks different from yesterday — your filters perform worse.

**What you do (no code typing):**
1. Click **Runtime → Run all** (or use the play buttons).
2. Adjust the **Drift Threshold** slider.
3. Click **Explain Change** to see plain-English hints.
4. Pick a **Playbook Action** and click **Apply**.
5. A **Decision Log** row is saved for your screenshot.

> If a widget looks stuck, just run the cell again once.


## What this lab is about

This is a no‑code lab to help you recognize data drift — when the data your model sees today looks different from the data it was trained on. In real ML Ops, noticing drift early helps you keep models safe and useful.

You won’t write code. You’ll just run cells and use a few interactive controls.

### The cybersecurity scenario (AURORA‑QR)
- Attackers send emails that contain an image with a QR code instead of a clickable link. The QR code, when scanned on a phone, opens a phishing site that steals credentials or installs malware.
- Why QR codes? Email security tools are good at scanning links and attachments in the email body. A QR code hides the destination inside the image, bypassing some checks. These emails often look clean and professional (written by AI), and may be followed by voice‑cloned phone calls to pressure help‑desk staff.
- What changes in the data when this happens?
  - More image‑only emails (screenshots, PNGs) without visible text links
  - More messages from new or unusual top‑level domains (like .zip or .mov)
  - Higher “AI‑likeness” in wording (fewer typos, more polished tone)
  - More MFA push events and help‑desk password resets (people getting tricked)
  - New TLS fingerprints (attackers using new infrastructure)

### “Today looks different from yesterday” — what does that mean?
- Security systems and ML models learn patterns from previous days (baseline). If today’s inputs shift — e.g., a sudden rise in image‑only emails — the model is seeing something it was not trained to expect.
- As a result, performance can get worse:
  - False negatives go up: more malicious emails slip through because the model isn’t tuned to QR‑in‑image tricks.
  - False positives can also change: if we react by turning up sensitivity, we might block more legitimate messages.

### What are “filters” here?
- Think of filters as rules and models used by email/security gateways to flag or block suspicious content. Examples:
  - Rule‑based filters: “block if image‑only AND unknown domain AND QR pattern detected.”
  - ML‑based filters: a model scores the probability an email is phishing based on signals (features) like domain reputation, formatting, attachment type, etc.
- When the attack style changes (data drift), the old filters no longer match the new reality. We must notice the change, explain it, and decide how to respond (retrain, add a new rule, or adjust thresholds).

### Why this matters in ML Ops
- Models are trained on yesterday’s world. Attackers and users change behavior.
- When inputs shift, model performance can degrade silently (more misses or more false alarms).
- ML Ops teams monitor for changes, explain what moved, and decide what to do next (retrain, add rules, tighten alerts, or just watch).

### What you’ll do
- Generate a baseline week and an “attack week” with synthetic security signals (e.g., QR‑code emails, new domains, MFA pushes).
- See how feature levels changed and which ones moved the most.
- Choose a playbook action and log your decision.

By the end, you’ll be able to explain: “What changed? Why might it have changed? What should we do about it?”


### Setup (what this does)
- Loads small helper libraries for charts and widgets.
- Defines two helper functions:
  - `badge(color, text)`: draws colored labels like Red/Yellow/Green.
  - `h2(text)`: displays section headers.

You don’t need to edit anything. Just run the cell so the notebook can create visual elements later.


In [14]:

#@title 🔧 Setup (just run)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
plt.rcParams["figure.figsize"] = (8,4)

def badge(color, text):
    return f'<span style="display:inline-block;padding:6px 10px;border-radius:8px;background:{color};color:white;font-weight:600">{text}</span>'

def h2(txt):
    return HTML(f"<h3 style='margin:6px 0 2px 0'>{txt}</h3>")


### Data generation (baseline vs. attack week)
This creates two small, synthetic datasets:
- Baseline: a normal work week.
- Attack: a week where phishing tactics increase (e.g., more image‑only emails with QR codes, new domain endings, more MFA push events, and help‑desk resets that might be social‑engineering).

Key fields you’ll see:
- `qr_attachment_ratio`, `image_only_email_ratio`, `new_tld_ratio_zip_mov`: email signals.
- `ai_likeness_score`: text looks more AI‑written.
- `mfa_push_events`, `mfa_denied_ratio`: authentication signals.
- `helpdesk_reset_calls`, `voice_clone_similarity`: social/voice attack signals.

Purpose: give you a controlled “before vs. after” so you can practice spotting drift.


In [15]:

#@title 📦 Generate Baseline & Attack (synthetic, embedded)
rng = np.random.default_rng(42)
n = 120  # ~one work week of hourly-ish slices

depts = ["Sales", "Finance", "HR", "IT", "Ops"]

def make_week(start_day, attack=False):
    dates = pd.date_range(start_day, periods=5, freq="D")
    times = pd.date_range("09:00", "17:00", freq="H").time  # business hours
    idx = []
    for d in dates:
        for t in times:
            idx.append(pd.Timestamp.combine(d, t))
    idx = np.array(idx)
    ts = rng.choice(idx, size=n, replace=True)
    ts.sort()

    df = pd.DataFrame({
        "timestamp": ts,
        "dept": rng.choice(depts, size=n, replace=True, p=[0.25,0.2,0.15,0.25,0.15]),
        "emails_total": rng.integers(200, 600, size=n),
        "unique_domains": rng.integers(30, 120, size=n),
        # security signals (0-1 ratios, small counts)
        "qr_attachment_ratio": rng.beta(1, 20, size=n),
        "image_only_email_ratio": rng.beta(1, 12, size=n),
        "new_tld_ratio_zip_mov": rng.beta(1, 18, size=n),
        "ai_likeness_score": rng.beta(2, 10, size=n),
        "typosquat_domain_ratio": rng.beta(1.2, 18, size=n),
        "ja3_new_fingerprint_ratio": rng.beta(1.5, 25, size=n),
        "mfa_push_events": rng.poisson(8, size=n),
        "mfa_denied_ratio": np.clip(rng.normal(0.06, 0.02, size=n), 0, 1),
        "failed_logins": rng.poisson(12, size=n),
        "helpdesk_reset_calls": rng.poisson(3, size=n),
        "voice_clone_similarity": np.clip(rng.normal(0.12, 0.04, size=n), 0, 1)
    })

    if attack:
        # AURORA-QR drift (dept-specific intensity so Dept filter is meaningful)
        size = len(df)
        # Base deltas
        d_qr   = rng.beta(2.5, 6, size=size) * 0.25
        d_img  = rng.beta(2.2, 6, size=size) * 0.20
        d_tld  = rng.beta(2.5, 6, size=size) * 0.25
        d_ai   = rng.beta(3,   5, size=size) * 0.35
        d_typo = rng.beta(2.2, 8, size=size) * 0.15
        d_ja3  = rng.beta(2.0,10, size=size) * 0.12
        d_mfa_push = rng.poisson(6,  size=size).astype(float)
        d_mfa_deny = rng.normal(0.05, 0.02, size=size)
        d_fail = rng.poisson(10, size=size).astype(float)
        d_help = rng.poisson(4,  size=size).astype(float)
        d_voice= rng.normal(0.25, 0.05, size=size)

        # Dept multipliers
        mult = {
            'qr_attachment_ratio':        np.ones(size),
            'image_only_email_ratio':     np.ones(size),
            'new_tld_ratio_zip_mov':      np.ones(size),
            'ai_likeness_score':          np.ones(size),
            'typosquat_domain_ratio':     np.ones(size),
            'ja3_new_fingerprint_ratio':  np.ones(size),
            'mfa_push_events':            np.ones(size),
            'mfa_denied_ratio':           np.ones(size),
            'failed_logins':              np.ones(size),
            'helpdesk_reset_calls':       np.ones(size),
            'voice_clone_similarity':     np.ones(size)
        }
        fin = (df['dept']=='Finance')
        it  = (df['dept']=='IT')
        hr  = (df['dept']=='HR')
        sales = (df['dept']=='Sales')
        ops = (df['dept']=='Ops')

        # Finance hit hardest by QR/image phishing and resets
        mult['qr_attachment_ratio'][fin]       *= 1.6
        mult['image_only_email_ratio'][fin]    *= 1.5
        mult['new_tld_ratio_zip_mov'][fin]     *= 1.4
        mult['mfa_push_events'][fin]           *= 1.3
        mult['helpdesk_reset_calls'][fin]      *= 1.4
        mult['ai_likeness_score'][fin]         *= 1.2

        # IT sees more infra/auth anomalies
        mult['ja3_new_fingerprint_ratio'][it]  *= 1.6
        mult['failed_logins'][it]              *= 1.4
        mult['mfa_denied_ratio'][it]           *= 1.2

        # HR targeted by voice-clone vishing
        mult['voice_clone_similarity'][hr]     *= 1.5
        mult['helpdesk_reset_calls'][hr]       *= 1.3

        # Sales more polished outreach and typosquats
        mult['ai_likeness_score'][sales]       *= 1.3
        mult['typosquat_domain_ratio'][sales]  *= 1.2

        # Ops moderate domain drift
        mult['new_tld_ratio_zip_mov'][ops]     *= 1.2
        mult['qr_attachment_ratio'][ops]       *= 1.1

        # Apply deltas with multipliers
        df["qr_attachment_ratio"]      += d_qr   * mult['qr_attachment_ratio']
        df["image_only_email_ratio"]   += d_img  * mult['image_only_email_ratio']
        df["new_tld_ratio_zip_mov"]    += d_tld  * mult['new_tld_ratio_zip_mov']
        df["ai_likeness_score"]        += d_ai   * mult['ai_likeness_score']
        df["typosquat_domain_ratio"]   += d_typo * mult['typosquat_domain_ratio']
        df["ja3_new_fingerprint_ratio"]+= d_ja3  * mult['ja3_new_fingerprint_ratio']
        df["mfa_push_events"]          += d_mfa_push * mult['mfa_push_events']
        df["mfa_denied_ratio"]          = np.clip(df["mfa_denied_ratio"] + d_mfa_deny * mult['mfa_denied_ratio'], 0, 1)
        df["failed_logins"]            += d_fail * mult['failed_logins']
        df["helpdesk_reset_calls"]     += d_help * mult['helpdesk_reset_calls']
        df["voice_clone_similarity"]     = np.clip(df["voice_clone_similarity"] + d_voice * mult['voice_clone_similarity'], 0, 1)

    for col in ["qr_attachment_ratio","image_only_email_ratio","new_tld_ratio_zip_mov",
                "ai_likeness_score","typosquat_domain_ratio","ja3_new_fingerprint_ratio",
                "mfa_denied_ratio","voice_clone_similarity"]:
        df[col] = df[col].clip(0,1)
    return df.sort_values("timestamp").reset_index(drop=True)

baseline = make_week("2025-03-03", attack=False)
attack = make_week("2025-03-10", attack=True)

KEYS = [
    "qr_attachment_ratio","image_only_email_ratio","new_tld_ratio_zip_mov",
    "ai_likeness_score","typosquat_domain_ratio","ja3_new_fingerprint_ratio",
    "mfa_push_events","mfa_denied_ratio","helpdesk_reset_calls","voice_clone_similarity"
]


/var/folders/0v/80zxmry158l85b2sy7ywwj5w0000gn/T/ipykernel_15210/3372025876.py:9: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  times = pd.date_range("09:00", "17:00", freq="H").time  # business hours
/var/folders/0v/80zxmry158l85b2sy7ywwj5w0000gn/T/ipykernel_15210/3372025876.py:9: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  times = pd.date_range("09:00", "17:00", freq="H").time  # business hours


### Drift dashboard (how to use it)
- Move the Drift Threshold slider to set how strict the detector is.
  - Green: low change
  - Yellow: medium change
  - Red: high change
- Use the Dept dropdown to focus on one department or view All.
- Click “Explain Change” to see plain‑English hints about what shifted most.

What’s happening underneath:
- The dashboard compares the average of each feature in the Baseline vs. Attack week.
- It computes a simple z‑score (how many standard deviations it moved) and maps that to colors.

This mimics an ML Ops monitor: notice, summarize, and explain drift quickly.


In [16]:

#@title 🚦 Drift Dashboard (run, then use the slider)
threshold = widgets.FloatSlider(
    value=1.0, min=0.2, max=3.0, step=0.1,
    description='Drift Threshold', readout_format='.1f', continuous_update=False
)
explain_btn = widgets.Button(description="Explain Change", button_style='info')
dept_dropdown = widgets.Dropdown(options=["All"]+depts, description="Dept")

dash_out = widgets.Output()
explain_out = widgets.Output()

def drift_scores(bdf, cdf, dept=None):
    if dept and dept != "All":
        bdf = bdf[bdf["dept"]==dept]
        cdf = cdf[cdf["dept"]==dept]
    means_b = bdf[KEYS].mean()
    means_c = cdf[KEYS].mean()
    std_b = bdf[KEYS].std().replace(0, np.nan)
    z = (means_c - means_b) / std_b
    z = z.fillna(0).abs()
    def label(val, t):
        if val < t: return ("Green", "#2ca02c")
        elif val < 2*t: return ("Yellow", "#ff7f0e")
        else: return ("Red", "#d62728")
    labels = {k: label(z[k], threshold.value) for k in KEYS}
    return means_b, means_c, z, labels

def plot_bars(means_b, means_c, z, threshold_value):
    idx = np.arange(len(KEYS))
    width = 0.38
    plt.figure()
    plt.bar(idx - width/2, means_b.values, width, label="Before")
    # Color "After" by severity using current threshold
    colors = []
    for k in KEYS:
        val = abs(float(z[k]))
        if val < threshold_value:
            colors.append('#2ca02c')  # green
        elif val < 2*threshold_value:
            colors.append('#ff7f0e')  # orange
        else:
            colors.append('#d62728')  # red
    plt.bar(idx + width/2, means_c.values, width, label="After", color=colors)
    plt.xticks(idx, [k.replace("_","\n") for k in KEYS], rotation=0)
    plt.legend()
    plt.title("Feature Levels — Before vs After (After colored by drift)")
    plt.tight_layout()

def render_dashboard(*_):
    with dash_out:
        clear_output()
        means_b, means_c, z, labels = drift_scores(baseline, attack, dept_dropdown.value)
        html = "<div style='display:flex;flex-wrap:wrap;gap:8px;margin:6px 0'>"
        for k in KEYS:
            txt, color = labels[k]
            html += badge(color, f"{k}: {txt}")
        html += "</div>"
        display(h2("Drift Badges"))
        display(HTML(html))
        display(h2("Before vs After"))
        plot_bars(means_b, means_c, z, threshold.value)
        plt.show()

def explain_action(btn):
    with explain_out:
        clear_output()
        _, _, z, _ = drift_scores(baseline, attack, dept_dropdown.value)
        top3 = sorted([(k, z[k]) for k in KEYS], key=lambda x: x[1], reverse=True)[:3]
        bullets = []
        for k, _score in top3:
            if "qr_attachment_ratio" in k:
                bullets.append("Image-only emails with **QR codes** increased — likely QR phishing.")
            elif "image_only_email_ratio" in k:
                bullets.append("More **image-only emails** (screenshots, no text) — suspicious.")
            elif "new_tld_ratio_zip_mov" in k:
                bullets.append("Unfamiliar **.zip/.mov** domains up — new phishing domains.")
            elif "ai_likeness_score" in k:
                bullets.append("Language looks **more AI-written** (smooth, fewer typos).")
            elif "ja3_new_fingerprint_ratio" in k:
                bullets.append("New **TLS/JA3** fingerprints — fresh attacker infra.")
            elif "mfa_push_events" in k:
                bullets.append("Spike in **MFA push** events — MFA fatigue attack.")
            elif "mfa_denied_ratio" in k:
                bullets.append("Higher **denied MFA** rate — people rejecting suspicious prompts.")
            elif "helpdesk_reset_calls" in k:
                bullets.append("More **help-desk resets** — vishing/voice-clone social engineering.")
            elif "voice_clone_similarity" in k:
                bullets.append("Rising **voice-clone similarity** — beware spoofed callers.")
            else:
                bullets.append(f"Change in **{k}** — investigate.")
        display(h2("Explain Change"))
        display(HTML("<ul>" + "".join([f"<li>{b}</li>" for b in bullets]) + "</ul>"))

threshold.observe(render_dashboard, names='value')
dept_dropdown.observe(render_dashboard, names='value')
explain_btn.on_click(explain_action)

display(widgets.HBox([threshold, dept_dropdown, explain_btn]))
display(dash_out)
display(explain_out)
render_dashboard()


Output()

Output()

### Playbook (making a decision)
Real teams don’t just observe—they act. Pick one action and click Apply:
- Retrain on last 7 days: adapts the model to new patterns; may raise noise.
- Add rule: block image‑only + unknown domain + QR: strong, fast protection; could block some legit emails.
- Lower alert threshold for Finance only: targeted sensitivity where risk is higher.
- Do nothing (monitor only): choose to watch if changes look temporary.

The notebook shows projected impact and saves a Decision Log row. This is how ML Ops captures why you chose an action.


In [17]:

#@title 🧭 Playbook — choose an action, then Apply
import uuid

action = widgets.RadioButtons(
    options=[
        "Retrain on last 7 days",
        "Add rule: block image-only + unknown domain + QR",
        "Lower alert threshold for Finance only",
        "Do nothing (monitor only)"
    ],
    description='Action:',
    layout=widgets.Layout(width='800px'),
    style={'description_width': 'initial'}
)
apply_btn = widgets.Button(description="Apply", button_style='success')
result_out = widgets.Output()

def simulate_effect(choice):
    if choice == "Retrain on last 7 days":
        return {"false_positives": "+2%", "false_negatives": "-15%", "note":"Better catch new phish; a bit noisier."}
    if choice == "Add rule: block image-only + unknown domain + QR":
        return {"false_positives": "+5%", "false_negatives": "-25%", "note":"Aggressive block; may hit some legit newsletters."}
    if choice == "Lower alert threshold for Finance only":
        return {"false_positives": "+1%", "false_negatives": "-10%", "note":"Targeted; least noisy; may miss some in other depts."}
    return {"false_positives": "0%", "false_negatives": "0%", "note":"Watching mode; risk remains."}

def on_apply(btn):
    with result_out:
        clear_output()
        _, _, z, _ = drift_scores(baseline, attack, dept_dropdown.value)
        top = sorted([(k, float(z[k])) for k in KEYS], key=lambda x: x[1], reverse=True)[:3]
        eff = simulate_effect(action.value)
        html_block = '''
        <div style="display:flex;gap:10px;margin:6px 0">
          {fp}
          {fn}
        </div>
        <div>{note}</div>
        '''
        display(h2("Projected Impact"))
        display(HTML(html_block.format(
            fp=badge('#1f77b4', 'False Positives: ' + eff['false_positives']),
            fn=badge('#9467bd', 'False Negatives: ' + eff['false_negatives']),
            note=eff['note']
        )))
        # Append to decision log (CSV in Colab workspace)
        log = pd.DataFrame([{
            "decision_id": str(uuid.uuid4())[:8],
            "dept_view": dept_dropdown.value,
            "top_changes": ";".join([f"{k}:{s:.2f}" for k,s in top]),
            "threshold": threshold.value,
            "action": action.value
        }])
        try:
            old = pd.read_csv("decision_log.csv")
            new = pd.concat([old, log], ignore_index=True)
        except Exception:
            new = log
        new.to_csv("decision_log.csv", index=False)
        display(h2("Decision Log (latest row)"))
        display(new.tail(1))

apply_btn.on_click(on_apply)
display(action, apply_btn, result_out)


RadioButtons(description='Action:', layout=Layout(width='800px'), options=('Retrain on last 7 days', 'Add rule…

Button(button_style='success', description='Apply', style=ButtonStyle())

Output()

---

## Key takeaways
- Data drift means “today’s data looks different from the training data,” which can silently degrade model performance.
- Good ML Ops treats drift detection as a continuous process: monitor → explain → decide → log.
- Actions have trade‑offs: fast rules can be noisy; retraining adapts but needs careful validation; targeted policies can reduce noise but may miss elsewhere.
- Decisions should be recorded. The Decision Log is your lightweight audit trail.

## Glossary (plain English)
- Feature: a measurable signal the model can “look at” (e.g., share of emails with QR codes).
- Baseline vs. Attack Week: a normal period vs. a period with suspicious changes.
- Z‑score: a way to say how big a change is compared to normal variation.
- False Positives / False Negatives: alarms that shouldn’t have fired vs. attacks that slipped through.
- Threshold: how sensitive you make detection. Higher sensitivity can catch more but alert more.

## Troubleshooting
- If widgets seem stuck, run the cell again once.
- If charts are blank, make sure you ran the Setup and Data Generation cells first.
- If you see file permission messages, don’t worry—saving the Decision Log locally is enough for this lab.

## For your write‑up
Briefly answer:
1) What changed the most and why might that indicate AURORA‑QR tactics?
2) Which action did you choose and why? What trade‑offs did you accept?
3) If you ran this weekly in production, what next signal would you add and why?



---

## ✅ What to turn in
1) A screenshot that shows the **Drift Badges** and **Before vs After** chart.  
2) A screenshot of the **Decision Log (latest row)** after you click **Apply**.  
3) Three sentences:
   - The biggest change was ____.
   - This likely means ____ attack behavior.
   - We chose ____ because it reduces ____ without too many false alarms.
